In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from hdfs import InsecureClient

HDFS_URL = "http://namenode:9870"
HDFS_BASE_PATH = "/hdfs-data"

client = InsecureClient(HDFS_URL, user="root")

In [ ]:
records = []

for country in client.list(HDFS_BASE_PATH):
    country_path = f"{HDFS_BASE_PATH}/{country}"
    for city in client.list(country_path):
        file_path = f"{country_path}/{city}/alerts.json"
        with client.read(file_path, encoding="utf-8") as reader:
            for line in reader:
                line = line.strip()
                if line:
                    records.append(json.loads(line))

df = pd.DataFrame(records)
df["event_time"] = pd.to_datetime(df["event_time"])
df = df.sort_values("event_time")
df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for city, group in df.groupby("city"):
    ax.plot(group["event_time"], group["temperature"], label=city)

ax.set_title("Evolution de la temperature au fil du temps")
ax.set_xlabel("Temps")
ax.set_ylabel("Temperature (C)")
ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

for city, group in df.groupby("city"):
    ax.plot(group["event_time"], group["windspeed"], label=city)

ax.set_title("Evolution de la vitesse du vent")
ax.set_xlabel("Temps")
ax.set_ylabel("Vitesse du vent (m/s)")
ax.legend()
plt.show()

In [ ]:
wind_counts = df["wind_alert_level"].value_counts().sort_index()
heat_counts = df["heat_alert_level"].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

wind_counts.plot(kind="bar", ax=axes[0])
axes[0].set_title("Nombre d'alertes vent par niveau")

heat_counts.plot(kind="bar", ax=axes[1])
axes[1].set_title("Nombre d'alertes chaleur par niveau")

plt.tight_layout()
plt.show()

In [ ]:
most_frequent_code = (
    df.groupby("country")["weather_code"]
    .agg(lambda codes: codes.value_counts().idxmax())
)
most_frequent_code